## Settting Up to Environment

# Dialogue System Enhancement Pipeline Example

## Introduction

This notebook demonstrates the **full pipeline** of the dialogue system enhancement project. We will go through the process of evaluating a pre-trained dialogue model and then fine-tuning it with **reinforcement learning (PPO)** to improve user satisfaction. Specifically, our reward signal is based on sentiment, encouraging the model to produce more positive and helpful responses.

**What this notebook covers:**
- Loading the **DailyDialog** dataset:contentReference[oaicite:10]{index=10} and preparing context-response pairs.
- Generating baseline responses using a pre-trained model (DialoGPT-small) and evaluating them (diversity and sentiment metrics).
- Fine-tuning the model with **Proximal Policy Optimization (PPO)** using sentiment-based rewards.
- Generating responses after reinforcement learning and comparing metrics to the baseline.

By the end, we'll see how the model's behavior changes with training aimed at making it more positive and satisfying for the user.

## Potential Applications

Improving a dialogue model's user satisfaction and positivity can be beneficial in various scenarios:
- **Customer Support Bots:** Ensuring the assistant responds in a friendly, positive manner to keep customer satisfaction high.
- **Mental Health Assistants:** Tuning the bot to give encouraging, empathetic responses to users expressing negative feelings.
- **Educational or Tutoring Agents:** Maintaining a positive tone to encourage and motivate learners.
- **Personal Companions (Chatbots):** Creating a more uplifting and engaging personality for general conversation.

In all these cases, a balance is needed between positivity and relevance. This project focuses on the positivity aspect, using sentiment as a proxy for user satisfaction.

## Setup

Before we start, make sure that you have a GPU runtime enabled (if using Colab or similar), as training the model with PPO is computationally intensive. We'll also assume the necessary libraries (Transformers, Datasets, TRL, etc.) are installed, for example via the provided Docker environment.

### Import Libraries and Module

First, we import the required libraries and our `DialogueRL_utils` module.


In [52]:
# Step 1: Environment Setup
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
import torch

# Check environment
print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.8.0+cu126
GPU available: True


## Loading The Dataset

In [2]:
from datasets import load_dataset

df = load_dataset("peandrew/dialy_dialogue_with_recoginized_concept_raw")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.14M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11118 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
df

DatasetDict({
    train: Dataset({
        features: ['dialog', 'act', 'emotion', 'mention', 'source', 'target'],
        num_rows: 11118
    })
    validation: Dataset({
        features: ['dialog', 'act', 'emotion', 'mention', 'source', 'target'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['dialog', 'act', 'emotion', 'mention', 'source', 'target'],
        num_rows: 1000
    })
})

## Inspecting The Data

In [8]:
import pandas as pd

# Convert a small portion to a DataFrame for display
df_train_sample = pd.DataFrame(df['train'][:])
df_train_sample


,dialog,act,emotion,mention,source,target
0,"[Say , Jim , how about going for a few beers a...","[3, 4, 2, 2, 2, 3, 4, 1, 3, 4]","[0, 0, 0, 0, 0, 0, 4, 4, 4, 4]","[[dinner, beer], [fitness, know], [will, mean,...","[dinner, beer]","[fun, dance, sound, exercise]"
1,"[Can you do push-ups ? , Of course I can . It...","[2, 1, 2, 2, 1, 1]","[0, 0, 6, 0, 0, 0]","[[push], [cake, push, believe, minute, piece],...",[push],"[everyday, exercise]"
2,"[Can you study with the radio on ? , No , I l...","[2, 1, 2, 1, 1]","[0, 0, 0, 0, 0]","[[radio, study], [listen, background, music], ...","[radio, study]","[player, buy, record]"
3,"[Are you all right ? , I will be all right so...","[2, 1, 1, 1]","[0, 0, 0, 0]","[[], [will, fall, watch, wire], [], [see]]","[will, fall, watch, wire]",[see]
4,"[Hey John , nice skates . Are they new ? , Ye...","[2, 1, 2, 1, 1, 2, 1, 3, 4]","[0, 0, 0, 0, 0, 6, 0, 6, 0]","[[skate], [league, ice, community, skate, play...",[skate],[see]
...,...,...,...,...,...,...
11113,"[Hello , I bought a pen in your shop just befo...","[1, 1, 1, 2, 3, 2, 1, 4, 1]","[0, 4, 0, 0, 0, 0, 0, 0, 4]","[[buy, shop, pen], [], [hotel, friend, break, ...","[buy, shop, pen]","[will, replace, receipt, shop]"
11114,"[Do you have any seats available ? , Yes . Th...","[2, 1, 2, 1, 3, 4]","[0, 0, 0, 0, 0, 4]","[[seat], [], [today], [pizza, recommend], [dro...",[seat],"[minute, wait]"
11115,"[Uncle Ben , how did the Forbidden City get th...","[2, 1, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 3, 4]","[0, 0, 6, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0]","[[city, forbid], [power, forbid, residence, pa...","[city, forbid]","[will, picture]"
11116,"[May I help you , sir ? , I want a pair of lo...","[2, 3, 4, 3]","[0, 0, 0, 0]","[[], [pair], [display], [need, size]]",[pair],"[need, size]"


In [9]:
print(df_train_sample.shape)
print(df_train_sample.columns)
print(df_train_sample.iloc[0])

(11118, 6)
Index(['dialog', 'act', 'emotion', 'mention', 'source', 'target'], dtype='object')
dialog     [Say , Jim , how about going for a few beers a...
act                           [3, 4, 2, 2, 2, 3, 4, 1, 3, 4]
emotion                       [0, 0, 0, 0, 0, 0, 4, 4, 4, 4]
mention    [[dinner, beer], [fitness, know], [will, mean,...
source                                        [dinner, beer]
target                         [fun, dance, sound, exercise]
Name: 0, dtype: object


## Loading the base model and Tokenization, Padding

In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Already added earlier
tokenizer.padding_side = "left"            # Add this line
model = AutoModelForCausalLM.from_pretrained(model_name)




## Generate Baseline Predictions

In [34]:
from tqdm import tqdm

# Generate baseline responses using DialoGPT
def generate_baseline_predictions(model, tokenizer, dataset, max_examples=100):
    prompts = []
    responses = []

    for i, sample in enumerate(dataset):
        if i >= max_examples:
            break

        input_text = sample["dialog"]  # Change this to "dialogue_text" if your dataset uses that
        prompts.append(input_text)

        # Tokenize and send to device
        encoded_input = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(model.device)

        # Generate response
        output = model.generate(
            **encoded_input,
            max_length=300,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode and store response
        decoded_response = tokenizer.decode(output[0], skip_special_tokens=True)
        responses.append(decoded_response)

    return prompts, responses


## Evaluate Sentiment Score of Responses

In [35]:
from transformers import pipeline

# Set up sentiment analysis model (binary POSITIVE/NEGATIVE)
sentiment_model = pipeline("sentiment-analysis", device=0)

def evaluate_sentiment(responses):
    results = sentiment_model(responses, truncation=True)
    pos_count = sum(1 for r in results if r["label"] == "POSITIVE")
    return pos_count / len(results), results  # return positive ratio and raw results


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


## Diversity Metrics (Distinct-1 and Distinct-2)

In [36]:
def distinct_n_gram(responses, n=1):
    all_ngrams = set()
    total_ngrams = 0

    for response in responses:
        tokens = response.split()
        ngrams = zip(*[tokens[i:] for i in range(n)])
        ngrams = list(ngrams)
        all_ngrams.update(ngrams)
        total_ngrams += len(ngrams)

    return len(all_ngrams) / total_ngrams if total_ngrams > 0 else 0.0


## Run the Baseline Evaluation

In [37]:
# Generate predictions (you can increase max_examples if GPU allows)
prompts, responses = generate_baseline_predictions(model, tokenizer, df["test"], max_examples=100)

# Sentiment evaluation
pos_ratio, sentiment_results = evaluate_sentiment(responses)
print(f"\n💡 Sentiment Positive Ratio: {pos_ratio:.2f}")

# Diversity metrics
distinct_1 = distinct_n_gram(responses, n=1)
distinct_2 = distinct_n_gram(responses, n=2)
print(f"💬 Distinct-1: {distinct_1:.3f}")
print(f"💬 Distinct-2: {distinct_2:.3f}")



💡 Sentiment Positive Ratio: 0.35
💬 Distinct-1: 0.400
💬 Distinct-2: 0.749


From the examples above, we can observe the baseline model's behavior. Some responses may be neutral or lack enthusiasm, depending on context. This is expected, as the model wasn't specifically tuned for positivity — it was trained on Reddit conversations:contentReference[oaicite:12]{index=12} which can be varied in tone.

Next, we'll proceed to **reinforcement learning fine-tuning** using PPO, to encourage the model to produce more positive and satisfying replies.


## Reinforcement Learning Fine-tuning (PPO)

We will fine-tune the model using Proximal Policy Optimization. The reward is based on the sentiment of the model's responses (as computed by our `compute_reward` function, which also penalizes very short answers).

### Training Configuration

For this example, we'll do a relatively short training (e.g., 100 PPO steps with batch size 8) for demonstration purposes. In a real scenario, we might run thousands of steps until convergence. We will use a subset of the training data to speed up the process.
